# Image autoencoders: compression and reconstruction

This notebook builds a convolutional autoencoder for 128 x 128 RGB images of apples and bananas. The encoder compresses each image into 128 learned values, and the decoder reconstructs an approximation of the processed input image.

The chapter follows the same explanation-then-code rhythm as the LSTM function-approximation notebook: inspect the data contract, trace every important tensor shape, test the model contract, train it, and visualize what it learned.

## Imports and reproducibility

PyTorch supplies the model, optimizer, dataset, and tensor operations. Torchvision reads image folders, applies preprocessing transforms, and arranges images into display grids. NumPy and Matplotlib support training summaries and visualization.

Fixed seeds make data shuffling and initial model weights repeatable.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import utils
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

np.random.seed(42)
torch.manual_seed(42)

## What are autoencoders and VAEs?

An **autoencoder** learns to represent an image using a smaller set of values. The **encoder** creates this compressed representation, called the latent representation. The **decoder** uses it to reconstruct an approximation of the original input.

A **variational autoencoder (VAE)** uses the same encoder to latent space to decoder structure, but its encoder describes a probability distribution by producing a mean and a variance rather than one fixed code. A VAE samples a latent representation from that distribution and gives it to the decoder. It still compares the reconstruction with the input, but it also adds a loss term that organizes the latent space. That organization makes it possible to sample new latent values and decode them into new images that resemble the training data.

This notebook builds a standard image autoencoder, not a VAE.

```mermaid
flowchart LR
    A[Processed RGB image<br/>3 x 128 x 128] --> E[Encoder]
    E --> Z[Latent representation<br/>128 values]
    Z --> D[Decoder]
    D --> R[Reconstructed RGB image<br/>3 x 128 x 128]
    A -. reconstruction target .-> R
```

## Preprocess the images

Each source image starts in the standard image layout `(height, width, color channels)`. RGB images have three channels: red, green, and blue. These transforms run in order before the dataset returns an image:

1. `Resize((128, 128))` gives every image the same spatial size so images can be stacked into batches. It rescales the entire image rather than cropping the outside. Bilinear interpolation computes each new pixel as a weighted average of nearby source pixels. Downscaling loses some fine detail, and forcing both dimensions to 128 can stretch a non-square image.
2. `ToTensor()` converts pixel values to floating point in `[0, 1]` and changes the layout to PyTorch's `(channels, height, width)`, producing `(3, 128, 128)`.
3. `Normalize(mean=0.5, std=0.5)` maps each RGB channel from `[0, 1]` to approximately `[-1, 1]`. Centered, similarly scaled inputs usually make neural-network optimization easier.

The image is still a two-dimensional pixel grid here; it is not flattened into a vector yet. From this point onward, **input image** means this processed tensor, not the raw file. The autoencoder learns to reconstruct the processed 128 x 128 RGB image and cannot recover detail lost during resizing.

In [ ]:
candidate_paths = [
    Path.cwd() / "data" / "train",
    Path.cwd() / "200_Autoencoders" / "data" / "train",
]
path_images = next((path for path in candidate_paths if path.exists()), candidate_paths[-1])

transform = transforms.Compose([
    transforms.Resize(
        (128, 128),
        interpolation=transforms.InterpolationMode.BILINEAR,
        antialias=True,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5),
    ),
])

print("Image directory:", path_images.resolve())

## Dataset and DataLoader

In the LSTM example, `TrigonometricDataset` was a custom `Dataset`: we supplied tensors and implemented `__len__` and `__getitem__` to define how one sample is read. `ImageFolder` is a ready-made dataset that already provides those methods for image files.

It expects subfolders such as `data/train/apples` and `data/train/bananas`. It scans the files, assigns a numeric label from each folder name, and applies `transform` whenever an image is requested. Each item is therefore `(image_tensor, label)`.

The label is useful for classification, but this autoencoder ignores it because its target is the input image itself. It reconstructs each processed image rather than predicting apple or banana.

`DataLoader` asks the dataset for samples, optionally shuffles their order, and stacks four images by adding a batch dimension:

- images: `(batch, channels, height, width) = (4, 3, 128, 128)`
- labels: `(4,)`

The loader does not decode image formats or resize pixels itself; `ImageFolder` and the transform pipeline do that before batching.

In [ ]:
dataset = ImageFolder(root=path_images, transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

images, labels = next(iter(dataloader))

print("Classes:", dataset.classes)
print("Number of images:", len(dataset))
print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Normalized value range:", (images.min().item(), images.max().item()))

## Inspect the processed inputs

Before training, convert a grid from normalized `[-1, 1]` values back to displayable `[0, 1]` values. This confirms that loading, resizing, channel order, batching, and normalization all behave as expected.

In [ ]:
def show_image(image, title=None):
    image = 0.5 * (image + 1)  # Change [-1, 1] back to [0, 1].
    image = image.clamp(0, 1)
    numpy_image = image.detach().cpu().numpy()

    plt.figure(figsize=(12, 4))
    plt.imshow(np.transpose(numpy_image, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


show_image(utils.make_grid(images), "Processed input images")

## Encoder: image to latent representation

The encoder receives one RGB image with shape `(3, 128, 128)`. A convolution uses a small sliding kernel to learn local visual features. Here, every kernel is 3 x 3 and moves one pixel at a time.

With stride 1 and no padding, each convolution reduces both spatial dimensions by two:

- input: `(3, 128, 128)`
- `Conv2d(3, 6, 3)`: `(6, 126, 126)`
- `Conv2d(6, 16, 3)`: `(16, 124, 124)`
- `Flatten()`: `16 x 124 x 124 = 246,016` values
- `Linear(...)`: `128` latent values

The three input channels correspond to RGB. The six and then sixteen output channels let the network learn progressively richer feature maps without changing the architecture's basic composition. A detailed understanding of convolution is not required for later generative-model chapters, but tracking the tensor shapes is essential.

In [ ]:
LATENT_DIMS = 128


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 3)
        self.conv2 = nn.Conv2d(6, 16, 3)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(16 * 124 * 124, LATENT_DIMS)

    # ReLU after each convolution adds nonlinearity so stacked layers can learn complex visual patterns instead of one linear mapping.
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

## Decoder: latent representation to image

The decoder reverses the shape changes. Its fully connected layer expands 128 latent values back to 246,016 values, which are reshaped into sixteen 124 x 124 feature maps.

A transposed convolution expands spatial dimensions when stride is 1 and padding is 0. Using ordinary `Conv2d` layers here would shrink `124` to `120`, so the reconstruction could not be compared with the 128 x 128 target.

- latent input: `128` values
- `Linear(...)` and reshape: `(16, 124, 124)`
- `ConvTranspose2d(16, 6, 3)`: `(6, 126, 126)`
- `ConvTranspose2d(6, 3, 3)`: `(3, 128, 128)`

The final three channels match the input's RGB channels.

In [ ]:
class Decoder(nn.Module):
    # `-> None` is an optional type hint indicating that __init__ returns no value.
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(LATENT_DIMS, 16 * 124 * 124)
        self.conv2 = nn.ConvTranspose2d(16, 6, 3)
        self.conv1 = nn.ConvTranspose2d(6, 3, 3)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()

    # ReLU after each convolution adds nonlinearity so stacked layers can learn complex visual patterns instead of one linear mapping.
    def forward(self, x):
        x = self.fc(x)
        x = x.reshape(-1, 16, 124, 124)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.conv1(x)
        x = self.relu(x)
        return x

## Combine the encoder and decoder

The `Autoencoder` class composes both modules. Its forward pass first compresses the image, then reconstructs it from the latent representation.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        return self.decoder(self.encoder(x))

## Check the model contract

A random batch provides a quick shape test before training. Four RGB inputs of shape `(4, 3, 128, 128)` must produce four reconstructions with exactly the same shape. The encoder must produce one 128-value latent vector per image.

This check catches incompatible input channels, incorrect flatten sizes, and decoder layers that shrink instead of expand the spatial dimensions.

In [ ]:
model = Autoencoder()
dummy_input = torch.rand((4, 3, 128, 128))

with torch.no_grad():
    dummy_latent = model.encoder(dummy_input)
    dummy_output = model(dummy_input)

print("Dummy input: ", dummy_input.shape)
print("Latent vectors:", dummy_latent.shape)
print("Model output:", dummy_output.shape)

assert dummy_latent.shape == (4, LATENT_DIMS)
assert dummy_output.shape == dummy_input.shape

## Loss, optimizer, and training

The autoencoder's target is the same processed tensor used as its input. Mean squared error measures the average squared pixel difference between the reconstruction and that target. Adam updates parameters in the encoder and decoder with learning rate `0.001`.

For each batch, training follows the usual PyTorch order:

1. create the reconstruction,
2. compare reconstruction and input,
3. clear old gradients,
4. backpropagate the new loss,
5. update the parameters.

Both tensors have shape `(batch, 3, 128, 128)`. The labels are deliberately ignored because this is reconstruction, not classification. The script trains for 30 epochs and reports the mean batch loss for each epoch.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
NUM_EPOCHS = 30
loss_history = []

model.train()
for epoch in range(NUM_EPOCHS):
    losses_epoch = []

    for batch_idx, (data, _) in enumerate(dataloader):
        output = model(data)

        # The model reconstructs the processed input, not the raw file on disk.
        loss = F.mse_loss(output, data)
        losses_epoch.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    mean_epoch_loss = np.mean(losses_epoch)
    loss_history.append(mean_epoch_loss)
    print(f"Epoch: {epoch:2d}  Loss: {mean_epoch_loss:.6f}")

## Training loss

A falling loss curve indicates that the reconstructed pixels are becoming closer to the processed input pixels. The curve reports training loss only, so it shows optimization progress rather than performance on unseen data.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), loss_history, color="tab:blue")
plt.title("Autoencoder training loss")
plt.xlabel("Epoch")
plt.ylabel("Mean MSE loss")
plt.grid(alpha=0.3)
plt.show()

## Compare original and reconstructed images

Switching to evaluation mode communicates that the model is being inspected rather than trained. `torch.no_grad()` avoids building a gradient graph during inference.

The first row contains processed input images; the second row contains their reconstructions. The model is reconstructing the resized, normalized tensors, not untouched source files.

In [ ]:
images, labels = next(iter(dataloader))

model.eval()
with torch.no_grad():
    reconstructed_images = model(images)
    reconstruction_mse = F.mse_loss(reconstructed_images, images).item()

comparison = torch.cat((images, reconstructed_images), dim=0)
comparison_grid = utils.make_grid(comparison, nrow=len(images))

show_image(comparison_grid, "Original images (top) and reconstructions (bottom)")
print(f"Batch reconstruction MSE: {reconstruction_mse:.6f}")

## Inspect the latent representations

The encoder produces 128 numbers for each image. Reshaping those values to `(1, 8, 16)` makes them displayable as a small single-channel grid, but this is only a visualization arrangement: the latent dimensions do not inherently have height and width.

Different patterns indicate that the encoder assigns different compact representations to different inputs.

In [ ]:
with torch.no_grad():
    latent_images = model.encoder(images)

latent_grids = latent_images.view(-1, 1, 8, 16)
show_image(
    utils.make_grid(latent_grids, nrow=len(images)),
    "Latent representations arranged as 8 x 16 grids",
)

print("Input images:", images.shape)
print("Latent vectors:", latent_images.shape)
print("Display grids:", latent_grids.shape)

## Compression rate

One processed RGB image contains `3 x 128 x 128 = 49,152` scalar values. Its latent representation contains 128 values.

The simple dimensional compression rate is

$$
\left(1 - \frac{\text{latent values}}{\text{input values}}\right) \times 100.
$$

This compares representation sizes only. It is not a file-compression measurement and does not include model parameters.

In [ ]:
image_size = images.shape[1] * images.shape[2] * images.shape[3]
compression_rate = (1 - LATENT_DIMS / image_size) * 100

print(f"Input values per image: {image_size:,}")
print(f"Latent values per image: {LATENT_DIMS:,}")
print(f"Dimensional compression rate: {compression_rate:.2f}%")

## Recap

- `ImageFolder` reads labeled folders, but an autoencoder uses each processed image as both input and target.
- The transform pipeline standardizes every sample as `(3, 128, 128)` with values approximately in `[-1, 1]`.
- The encoder reduces an image to 128 learned values; the decoder expands those values back to the original tensor shape.
- Transposed convolutions are required here because the decoder must expand `124 -> 126 -> 128`.
- MSE trains the complete encoder-decoder pipeline by measuring pixel-level reconstruction error.
- Reshaping a latent vector to 8 x 16 is useful for display, but does not turn it into a true spatial feature map.
- The 99.74% compression figure compares scalar dimensions, not file sizes or total model memory.

A useful next experiment is to compare reconstruction quality for several values of `LATENT_DIMS`. Smaller latent spaces force stronger compression; larger ones give the model more capacity to preserve detail.

## Optional custom diagrams

These prompts can be pasted into Copilot image generation. Save the generated files under `200_Autoencoders/assets/`, then add standard Markdown image links near the matching sections.

### 1. Autoencoder tensor-shape flow

Suggested filename: `autoencoder_tensor_shape_flow.png`

> Create a precise educational infographic showing the tensor-shape flow through a convolutional image autoencoder. Use a clean white background and a wide 16:9 composition. From left to right show: a realistic square RGB fruit image labeled "Input 3 x 128 x 128"; an encoder block labeled "Conv2d: 3 to 6, 126 x 126"; a second encoder block labeled "Conv2d: 6 to 16, 124 x 124"; a narrow central bottleneck labeled "Latent vector: 128 values"; a decoder block labeled "Linear + reshape: 16 x 124 x 124"; a transposed-convolution block labeled "16 to 6, 126 x 126"; and a final transposed-convolution block leading to "Reconstruction 3 x 128 x 128". Use restrained blue for encoder stages, amber for the bottleneck, green for decoder stages, thin directional arrows, crisp typography, and no logos or decorative filler. Make every shape label large and legible.

### 2. Standard autoencoder versus VAE

Suggested filename: `autoencoder_vs_vae.png`

> Create a side-by-side educational comparison of a standard autoencoder and a variational autoencoder on a white background in a wide notebook-friendly layout. On the left, show "Standard autoencoder": input image to encoder to one fixed latent vector z to decoder to reconstruction. On the right, show "Variational autoencoder": input image to encoder, branching into mean mu and log variance, then a sampling step labeled "z = mu + sigma epsilon", then decoder to reconstruction. Beneath the VAE add two compact loss labels: "reconstruction loss" and "KL divergence". Use consistent layer symbols, clear arrows, high-contrast labels, restrained blue/green/gold colors, and no logos, equations beyond the sampling expression, or decorative background elements.

### 3. Reconstruction target and MSE

Suggested filename: `autoencoder_reconstruction_loss.png`

> Create a three-panel educational diagram explaining autoencoder reconstruction loss for RGB images. Panel one: a processed 128 x 128 fruit image labeled "Input and target" with a small note "normalized pixels". Panel two: the reconstructed image labeled "Model output". Panel three: a pixelwise error heatmap labeled "Squared difference", feeding into a compact formula "MSE = mean((output - input)^2)". Use matching image framing so corresponding pixels align visually, a perceptually clear heatmap, concise labels, a white background, and a wide 16:9 layout suitable for a Jupyter notebook. Avoid logos, decorative text, and photorealistic UI mockups.